# Run the study on Colab

Runs every experiment in this repo, in order. Results go to Google Drive
(`MyDrive/rag-context-attention/`), and every long script saves as it goes, so if the
session disconnects, **reconnect, re-run cells 1–3, then re-run the cell that was
interrupted**: finished work is loaded, not redone.

**GPU** (Runtime → Change runtime type):

| Section | GPU | Rough time |
|---|---|---|
| 5. Exp 1C tiny GPT | T4 is enough (free tier) | ~1 h |
| 6. Exp 1A/1B prompt length | **A100** (fastest) or L4 | A100 ~2–3 h, L4 ~6–8 h |
| 7. Exp 1 attention | A100 or L4 | ~2–4 h |
| 8. Exp 2 iterative retrieval | A100 or L4 | ~1 h |
| 9. Analysis | any / CPU | minutes |

Times are guesses: the debug runs (`--limit`) in each section print timings you can scale up.
Keep the browser tab open during long cells; Colab Pro disconnects closed tabs.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Drive, code and folders

Clones the repo (or pulls it if already cloned this session), then links `data/raw`,
`data/processed` and `results` to Drive so they survive disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess
REPO = '/content/rag-context-attention'
DRIVE = '/content/drive/MyDrive/rag-context-attention'

if os.path.exists(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/MitushiG/rag-context-attention.git', REPO], check=True)
os.chdir(REPO)

for sub in ['data/raw', 'data/processed', 'data/samples', 'results']:
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)
    os.makedirs(os.path.dirname(sub) or '.', exist_ok=True)
    if sub == 'data/samples' and os.path.isdir(sub) and not os.path.islink(sub):
        continue  # sample ids already committed to git: use those
    if os.path.islink(sub):
        continue
    if os.path.isdir(sub):
        shutil.copytree(sub, f'{DRIVE}/{sub}', dirs_exist_ok=True)  # keep committed files (e.g. summary.md)
        shutil.rmtree(sub)
    os.symlink(f'{DRIVE}/{sub}', sub)

!git log --oneline -3
!ls -la data results

## 3. Install

vLLM first (it pins torch), then the rest. Re-run this after every reconnect.

In [ ]:
!pip install -q vllm
!pip install -q -r requirements.txt && pip install -q -e .
import torch; print(torch.__version__, torch.cuda.is_available())

## 4. Data (once)

NQ downloads automatically. MuSiQue is fetched with its official `download_data.sh`
(a Google Drive zip via `gdown`). If that download fails, download
`musique_ans_v1.0_dev.jsonl` by hand from https://github.com/StonyBrookNLP/musique and
upload it to `MyDrive/rag-context-attention/data/raw/`.

In [ ]:
import os
if not os.path.exists('data/raw/musique_ans_v1.0_dev.jsonl'):
    !pip install -q gdown
    !rm -rf /content/musique && git clone -q https://github.com/StonyBrookNLP/musique.git /content/musique
    !cd /content/musique && bash download_data.sh
    !cp /content/musique/data/musique_ans_v1.0_dev.jsonl data/raw/
!ls -la data/raw

In [ ]:
if not os.path.exists('data/processed/corpus.jsonl'):
    !python scripts/prepare_data.py
!wc -l data/samples/*.txt data/processed/*.jsonl

**Pre-registration:** the sampled question ids are now in Drive at
`MyDrive/rag-context-attention/data/samples/`. Commit them to the repo (`data/samples/*.txt`)
before looking at any results. Sampling is seeded (13), so re-running cell 4 gives the same ids.

In [ ]:
!python -m pytest -q

## 5. Exp 1C: tiny GPT (T4 is enough)

4 variants × 3 seeds. Each finished model is saved to `results/tiny/`, and a re-run skips it.
Watch for `WARNING: final training accuracy < 0.95`: if it appears, the default 8000 steps
weren't enough. Note that before comparing variants.

In [ ]:
!python foundations/tiny_recall.py --sweep

## 6. Exp 1A/1B: prompt length (A100 or L4)

Debug on 5 questions first: check the EM numbers look sane (gold_only should be high and
closed_book low) and note how long it takes. Debug output goes to `results/exp1/parts_limit5/`
and does not mix with the full run.

In [ ]:
!python scripts/run_exp1_length.py --limit 5

In [ ]:
!python scripts/run_exp1_length.py

## 7. Exp 1 attention (after section 6, transformers not vLLM)

`--check` first compares the memory-efficient probe with eager attention on short inputs
and stops if they disagree. `attention.jsonl` is appended row by row, so it resumes too.

In [ ]:
!python scripts/run_exp1_attention.py --check --limit 2

In [ ]:
!python scripts/run_exp1_attention.py

## 8. Exp 2: single-shot vs CoT vs IRCoT

Debug on 20 questions per dataset, then **read some reasoning chains** (next cell) before the full run.

In [ ]:
!python scripts/run_exp2_iterative.py --limit 20

In [ ]:
import json
rows = [json.loads(l) for l in open('results/exp2/exp2.jsonl')]
for r in [r for r in rows if r['method'] == 'ircot' and r['dataset'] == 'musique'][:5]:
    print(json.dumps({k: v for k, v in r.items() if k not in ('context_ids',)}, indent=1)[:2500], '\n' + '-' * 80)

In [ ]:
!python scripts/run_exp2_iterative.py

## 9. Analysis

In [ ]:
!python scripts/analyze.py
from IPython.display import Markdown
Markdown(open('results/summary.md').read())